# MetroPT-3 data audit

This notebook measures the dataset before any sequence architecture or resampling interval is fixed. It does not train a model or inspect final-holdout predictions. A standard CPU runtime is sufficient.

In [ ]:
from pathlib import Path
import os
import subprocess

repo = Path('/content/metropt3-predictive-maintenance')
if not repo.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'investigation/temporal-validation',
        '--single-branch',
        'https://github.com/SahilBh01r1769/metropt3-predictive-maintenance.git',
        str(repo),
    ], check=True)
os.chdir(repo)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Auditing revision:', revision)

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['python', 'scripts/download_data.py'], check=True)
environment = os.environ.copy()
environment['PYTHONPATH'] = 'src'
subprocess.run([
    'python', '-m', 'metropt3.cli', 'audit',
    '--output', 'artifacts/data_audit.json',
], check=True, env=environment)

In [ ]:
import json
import zipfile

context = {'git_revision': revision}
Path('artifacts/audit_context.json').write_text(json.dumps(context, indent=2))
bundle = '/content/metropt_audit_bundle.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write('artifacts/data_audit.json', arcname='data_audit.json')
    archive.write('artifacts/audit_context.json', arcname='audit_context.json')
print('Created:', bundle)
from google.colab import files
files.download(bundle)